# Notebook 7: Performance Comparison — NW-flex vs BWA-MEM
This notebook compares NW-flex against BWA-MEM, a widely used short-read aligner, on simulated reads from real human STR loci. Where Notebook 6 demonstrated the alignment workflow on a synthetic single locus with random flanks, here we approach real-world conditions: the loci come from a panel of human STR sites in hg38, carrying their genuine flank sequences and repeat motifs. Reads are tiled across haplotypes whose repeat count differs from the reference, and we ask under what conditions each method recovers the haplotype's repeat-region length.

**A note on BWA-MEM.** BWA-MEM is a two-stage aligner. It first performs a *global seed search* of the reference, finding maximal exact matches anywhere in the genome. It then performs *local alignment* by Smith-Waterman extension around the most promising seeds, scored with an affine-gap scheme. Soft-clipping arises in this second stage: when extending past a seed costs more than declaring the read end "clipped", BWA-MEM emits an `S` op in the CIGAR.

## Overview

In this notebook we:

1. **Build the simulation machinery** — locus, haplotype, and read tiling — on top of a panel of real STR loci drawn from hg38 (selected to be pure and locally-isolated).

2. **Use a BWA-MEM-compatible scoring scheme.** NW-flex's defaults module ships with several alternative scoring schemes; for this notebook we select the one whose match, mismatch, gap-open, and gap-extend values match BWA-MEM's defaults, so the two methods see the same scoring landscape on every read.

3. **Set up three alignment configurations**: BWA-MEM at standard parameters; BWA-MEM with the soft-clip penalty raised so high that clipping never improves the score; and NW-flex with the STR-aware EP pattern from Notebook 4. For BWA-MEM we align both orientations and take the better-scoring strand so the comparison does not penalise BWA-MEM for orientation artifacts.

4. **Run three comparisons**, each changing one thing about the simulation:
   - **Length variation only** — haplotype repeat count varies, flanks unchanged.
   - **One SNV in the flank** — same sweep with a single base change one bp outside the repeat boundary.
   - **Compound repeat** — the locus is two adjacent motifs joined by a short interrupting sequence; both counts vary independently.

5. **Tabulate correctness** for each method as a function of read flank extent and the haplotype's $\Delta$, and present each result as a heatmap with one panel per method.

The headline result is structural: when the only difference between haplotype and reference is the repeat counts, NW-flex's EP pattern is guaranteed to find the optimum, while a clip-based aligner is at the mercy of its scoring tradeoffs at the boundary. Flank variations expose all aligners to score optimizations that may alter the measured repeat length.

This notebook builds on:

- **Notebook 4**: STR specialization and the phase-preserving EP pattern.
- **Notebook 6**: STR locus simulation and pileup.

The notebook runs on the committed panel TSV plus `bwa` and `samtools` on `PATH`. If those are missing, the BWA cells skip cleanly with an installation hint.

## Setup and imports

### Imports

Standard scientific-Python imports plus the core NW-flex pieces this notebook builds on. Notebook-specific helpers (panel loading, BWA wrapper, CIGAR decoder) will be added here as the cells that need them are written.

In [15]:
# 🧙 Notebook magic: autoreload modules
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

# Core NW-flex pieces
from nwflex.dp_core import FlexInput
from nwflex.fast import run_flex_dp_fast
from nwflex.ep_patterns import build_EP_STR_phase
from nwflex.repeats import STRLocus

# Simulation harness (this notebook)
from nwflex.simulation import build_locus_from_panel, build_haplotype, tile_reads

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Scoring scheme

NW-flex ships several alternative scoring schemes in `nwflex.default`, each a `(score_matrix, gap_open, gap_extend)` triple keyed by name. The original NW-flex default uses round-number values chosen for legibility in the teaching notebooks; the additional schemes target compatibility with external aligners.

For this notebook we select the **BWA-MEM-compatible** scheme so that NW-flex and BWA-MEM see matches, mismatches, and gaps under identical costs. With matched scoring, any disagreement between the two methods reflects what each algorithm prefers given the same alignment landscape, not a difference in the landscape itself.

In [16]:
from nwflex.default import get_default_scoring

# BWA-MEM-compatible scoring (+1 / -4 / -6 / -1).
score_matrix, gap_open, gap_extend, a2i = get_default_scoring("bwa_mem")

### BWA-MEM and samtools availability

The BWA arms of this notebook shell out to `bwa` and `samtools`, which must be on `PATH`. The cell below checks both; if either is missing, the BWA cells skip cleanly with an installation hint.

In [17]:
import shutil

required = ("bwa", "samtools")
missing = [t for t in required if shutil.which(t) is None]
if missing:
    print(f"Missing on PATH: {', '.join(missing)}")
    print(f"Install with: conda install -c bioconda {' '.join(missing)}")
    print("BWA-MEM cells will be skipped if these are unavailable.")
else:
    print(f"Found on PATH: {', '.join(required)}")

Found on PATH: bwa, samtools


### Input panel

The simulation draws each locus from a panel of human STR sites stored at `data/hg38_motif_sample_K100.tsv`. The panel was built using [Tandem Repeat Finder (TRF)](https://tandem.bu.edu/trf/trf.html) run on hg38 and the results  filtered to keep only loci that are *pure* (a clean run of a single canonical motif) and *locally isolated* (no other repeats in the surrounding flank window). Each row gives a genomic location, the motif and tract as found in the reference, and the flanking sequences we use to build the simulation reference. The panel has example loci for mono, di and trinucleotide motifs.

Schema:

| Column | Meaning |
|---|---|
| `pind` | panel index (locus identifier) |
| `chr`, `start_38`, `stop_38`, `strand` | hg38 coordinates of the repeat tract |
| `type` | canonical motif (e.g. `A`, `AC`, `AGG`) |
| `lflank`, `rflank` | left and right flank sequences (500bp) |
| `ms_seq` | repeat tract as it appears in hg38 |
| `ref_score_per_base` | per-base purity score (1.0 = perfect tract) |

> *TODO:* a separate appendix notebook (or script) covering the TRF run, the purity / isolation filters, and the panel-construction steps would make the data reproducible from raw inputs. Out of scope for this notebook; link here when written.

In [18]:
from pathlib import Path
import pandas as pd

PANEL_PATH = Path("../data/hg38_motif_sample_K100.tsv")
panel = pd.read_csv(PANEL_PATH, sep="\t")

print(f"Loaded {len(panel):,} loci from {PANEL_PATH}")
print(f"Motif-length breakdown: {panel['type'].str.len().value_counts().sort_index().to_dict()}")
panel.head()

Loaded 6,900 loci from ../data/hg38_motif_sample_K100.tsv
Motif-length breakdown: {1: 400, 2: 1080, 3: 5420}


,pind,chr,start_38,stop_38,strand,type,lflank,rflank,ms_seq,ref_score_per_base
0,0,chr1,36351,36364,+,A,CCATCTCTGGGCCCAGAATGACCCACTGGAGACCTTACAGCTCTCC...,CCCAGCCTGGCGGAAAGAATTTAAATTATAAAAACTTAGAAGTATG...,AAAAAAAAAAAAA,1.0
1,1,chr1,51864,51877,+,A,GTGGGGGTTGAGTTTCACTTTATTTAAAGTGAGTCTTAATCCTCCA...,GAAGATTGATCAGAGAGTACCTCCCCTAAGGGTACATGCAGATAAA...,AAAAAAAAAAAAA,1.0
2,2,chr1,71175,71186,+,A,AGTATATTACTTGGATCCATCTATGTCATTTTCCATGGTTAATGTT...,CCTTAACAAATGATTCTGACAAATATCTTCTCTTTCCAGGGAGAAT...,AAAAAAAAAAA,1.0
3,3,chr1,77174,77195,+,A,CTTCATGTCTAAAACACCGAGAGAGGCACTCTTATGCATTGTTGGT...,GGAAAATAACCAAATGACAATTAGTGAGTACTACTTGCAAAACTTG...,AAAAAAAAAAAAAAAAAAAAA,1.0
4,4,chr1,108545,108561,+,A,TGTACCATGCTCCTCCTTAATCATTCTGAGGTTACATCTTAAGTCC...,GAATGGAGAGAATGCTACATGAGAGAAAGGATCTTATCTATCATGT...,AAAAAAAAAAAAAAAA,1.0


## Simulation setup

We construct the simulation in three stages: locus, haplotype, reads.

### Locus

The locus comes from the panel: a real flank pair, a real repeat motif, and a chosen reference repeat count $N$. Together these give the reference sequence $X = A \cdot R^N \cdot B$. We trim the genomic flanks to a fixed length and to a clean motif-edge boundary so the flank/repeat boundary is unambiguous. Tandem Repeat Finder chooses the motif rotation so the repeat tract begins on a full motif copy, but it does not require the tract to end on one — any trailing partial copy then sits at the start of the right flank, where it would read as an extension of the repeat. We trim both flanks Just In Case.

In [19]:
LOCUS_FLANK_LEN = 200

# Pick a representative trinucleotide locus from the panel.
row = panel[panel["type"].str.len() == 3].iloc[0]
ref_n = len(row["ms_seq"]) // len(row["type"])

locus = build_locus_from_panel(
    motif=row["type"],
    panel_lflank=row["lflank"],
    panel_rflank=row["rflank"],
    ref_n=ref_n,
    flank_len=LOCUS_FLANK_LEN,
)

print(f"pind={row['pind']}  motif={locus.R!r}  ref N={locus.N}")
print(f"  X length: {locus.n}  ({len(locus.A)} flank + {locus.k * locus.N} repeat + {len(locus.B)} flank)")
print(f"  z_start={locus.s}  z_end={locus.e}")
print(f"  A[-10:] = {locus.A[-10:]}   (last base must not be {locus.R[-1]!r})")
print(f"  B[:10]  = {locus.B[:10]}   (first base must not be {locus.R[0]!r})")

pind=1480  motif='AAC'  ref N=5
  X length: 415  (200 flank + 15 repeat + 200 flank)
  z_start=200  z_end=215
  A[-10:] = GCTGCTTATA   (last base must not be 'C')
  B[:10]  = CATTGACAGG   (first base must not be 'A')


### Haplotype

The haplotype is the sequence we sample reads from. It shares the reference's flanks and motif but has $N + \Delta$ motif copies, for any integer $\Delta$ (including zero). What follows asks how reads of the haplotype align against the reference as $\Delta$ varies.

In [20]:
DELTA = 1  # haplotype has one extra repeat copy vs the locus

hap = build_haplotype(locus, delta=DELTA)
rflank_len = len(hap.sequence) - hap.flank_len - hap.body_len

print(f"locus N={locus.N}  →  haplotype N={locus.N + DELTA}")
print(f"  hap length: {len(hap.sequence)}  ({hap.flank_len} flank + {hap.body_len} repeat + {rflank_len} flank)")

locus N=5  →  haplotype N=6
  hap length: 418  (200 flank + 18 repeat + 200 flank)


### Reads

Each read is a fixed-length contiguous segment of the haplotype. We generate one read at every starting position whose segment covers at least $K$ bases of each flank: this guarantees the read spans the repeat and gives the alignment flank context on both sides. The number of flank bases a read covers on each side is its *flank extent*.

In [21]:
READ_LEN = 150
K_MIN_FLANK = 1

reads = tile_reads(hap, read_len=READ_LEN, k_min_flank=K_MIN_FLANK)
lflank_extents = sorted({r.lflank_extent for r in reads})
rflank_extents = sorted({r.rflank_extent for r in reads})

print(f"Tiled {len(reads)} reads (read_len={READ_LEN}, k_min_flank={K_MIN_FLANK})")
print(f"  lflank_extent: {lflank_extents[0]}..{lflank_extents[-1]}")
print(f"  rflank_extent: {rflank_extents[0]}..{rflank_extents[-1]}")

Tiled 131 reads (read_len=150, k_min_flank=1)
  lflank_extent: 1..131
  rflank_extent: 1..131


## Three alignment configurations
We align each read against the locus reference under three configurations:
1. BWA-MEM at standard parameters
2. BWA-MEM no soft-clipping
3. NW-flex with BWA-MEM parameters

### BWA-MEM configurations
In the first two configurations, we generate a BWA-MEM index for the reference sequence and run `bwa mem` on each read. The first step of BWA-MEM's two-stage process — the global seed search — should be easy in this context. The second step performs a local-alignment by Smith-Waterman extension.

The difference between the first two configurations is the soft-clip penalty. In the first, we use BWA-MEM's default parameters, which allow soft-clipping when it improves the score. In the second, we raise the soft-clip penalty so high that clipping never improves the score (`-L 500`), effectively disabling clipping. This isolates the soft-clip mechanism: a regime where both BWA arms fail is a structural failure, not a clipping artifact.

In [22]:
from nwflex.simulation import align_bwa

# Three reads spanning the boundary range.
illustrative_idx = [0, len(reads) // 2, len(reads) - 1]
illustrative_reads = [reads[i] for i in illustrative_idx]

res_std    = align_bwa(locus.X, illustrative_reads, no_clip=False)
res_noclip = align_bwa(locus.X, illustrative_reads, no_clip=True)

print("BWA-MEM on three illustrative reads — std (defaults) vs no-clip (-L 500):")
pd.DataFrame({
    ("read",   ""):        illustrative_idx,
    ("lflank", ""):        [r.lflank_extent for r in illustrative_reads],
    ("rflank", ""):        [r.rflank_extent for r in illustrative_reads],
    ("cigar",  "std"):     [x.cigar for x in res_std],
    ("score",  "std"):     [x.score for x in res_std],
    ("cigar",  "no-clip"): [x.cigar for x in res_noclip],
    ("score",  "no-clip"): [x.score for x in res_noclip],
})

BWA-MEM on three illustrative reads — std (defaults) vs no-clip (-L 500):


,read,lflank,rflank,cigar,score,cigar,score
,,,,std,std,no-clip,no-clip
0,0,131,1,146M4S,146,131M3I16M,146
1,65,66,66,66M3I81M,138,66M3I81M,138
2,130,1,131,4S146M,146,1M3I146M,146


### A closer look at the alignments

The CIGARs above tell the story but it's easier to see the base-level details. The helper `render_zoom` shows the read aligned to the reference, with `|` marking the repeat-zone boundaries, `---` columns where one side carries a gap, an `I`/`D` marker line under any inserted or deleted columns, and bracketed `[...]` for soft-clipped read bases.

Three things to notice as you read down the page:

- The boundary reads (idx 0 and 130) under **std** soft-clip the extra `AAC` motif (plus the lone flank base) — the alignment consumes only the reference's repeat zone and writes the rest off as clipped.
- The same reads under **no-clip** insert the extra motif (`---` on the ref line, `III` underneath) at the repeat-zone boundary.
- The centered read (idx 65) inserts the extra motif under both configurations: with flank context on both sides, the SW extender has no incentive to clip.

In [23]:
from nwflex.simulation import render_zoom

for ix, r, s, n in zip(illustrative_idx, illustrative_reads, res_std, res_noclip):
    print(f"Read idx={ix}   lflank={r.lflank_extent:>3}   rflank={r.rflank_extent:<3}")
    for label, hit in (("STD   ", s), ("NOCLIP", n)):
        print(f"  {label}  {hit.cigar}  (pos={hit.pos}, AS={hit.score})")
        for line in render_zoom(
            locus.X, r.sequence, hit.pos, hit.cigar, locus.s, locus.e,
        ).splitlines():
            print(f"    {line}")
        print()


Read idx=0   lflank=131   rflank=1  
  STD     146M4S  (pos=70, AS=146)
    ref :  AATCTGCTGCTTATA|AACAACAACAACAAC  
    read:  AATCTGCTGCTTATA|AACAACAACAACAAC[AACC]

  NOCLIP  131M3I16M  (pos=70, AS=146)
    ref :  AATCTGCTGCTTATA|---AACAACAACAACAAC|C
    read:  AATCTGCTGCTTATA|AACAACAACAACAACAAC|C
                           III                 

Read idx=65   lflank= 66   rflank=66 
  STD     66M3I81M  (pos=135, AS=138)
    ref :  AATCTGCTGCTTATA|---AACAACAACAACAAC|CATTGACAGGCACAA
    read:  AATCTGCTGCTTATA|AACAACAACAACAACAAC|CATTGACAGGCACAA
                           III                               

  NOCLIP  66M3I81M  (pos=135, AS=138)
    ref :  AATCTGCTGCTTATA|---AACAACAACAACAAC|CATTGACAGGCACAA
    read:  AATCTGCTGCTTATA|AACAACAACAACAACAAC|CATTGACAGGCACAA
                           III                               

Read idx=130   lflank=  1   rflank=131
  STD     4S146M  (pos=201, AS=146)
    ref :        AACAACAACAACAAC|CATTGACAGGCACAA
    read:  [AAAC]AACAACAACAACAAC|CATTG


### NW-flex configuration
The third configuration replaces BWA-MEM with NW-flex. Where BWA-MEM seeds against the locus reference and extends locally, NW-flex performs a global alignment against an extended reference $X' = A \cdot R^{3N} \cdot B$ — the same flanks as the locus reference, with three times as many motif copies. The STR-aware EP pattern from Notebook 4 contracts that repeat run to any allowed haplotype count in a single pass. We pick $3N$ so the sweep ($N + \Delta$ for small $\Delta$, either sign) sits well inside the allowed range.

In [24]:
from nwflex.aligners import RefAligner

# Build the extended NW-flex reference: same flanks as the locus, three times the motif copies.
nwflex_locus = STRLocus(A=locus.A, R=locus.R, N=3 * locus.N, B=locus.B)

# Build the STR-aware EP pattern (Notebook 4); contracts the 3N run to any allowed haplotype count in one pass.
EP = build_EP_STR_phase(nwflex_locus.n, nwflex_locus.s, nwflex_locus.e, nwflex_locus.k)

# Construct the aligner with BWA-MEM-compatible scoring (matched to the BWA cell above).
aligner = RefAligner(
    ref=nwflex_locus.X,
    extra_predecessors=EP,
    score_matrix=score_matrix,
    gap_open=gap_open,
    gap_extend=gap_extend,
    alphabet_to_index=a2i,
    free_X=True,
    free_Y=True,
    max_read_length=READ_LEN + 5,
)

# Run NW-flex on the same three illustrative reads.
res_nwflex = [aligner.align_simple(reads[i].sequence) for i in illustrative_idx]

print("NW-flex on three illustrative reads (3N reference, STR-aware EP):")
pd.DataFrame({
    "read":   illustrative_idx,
    "lflank": [r.lflank_extent for r in illustrative_reads],
    "rflank": [r.rflank_extent for r in illustrative_reads],
    "cigar":  [a["cigar"] for a in res_nwflex],
    "score":  [int(a["score"]) for a in res_nwflex],
})

NW-flex on three illustrative reads (3N reference, STR-aware EP):


,read,lflank,rflank,cigar,score
0,0,131,1,131M27N19M,150
1,65,66,66,66M27N84M,150
2,130,1,131,1M27N149M,150


Visually, NW-flex's alignment looks much like BWA-MEM's no-clip version, except the extra motif is cleanly inserted at the boundary for all reads, including the boundary ones. The EP pattern's repeat run allows NW-flex to skip unutilized repeat copies without paying a penalty, so the alignment is not forced to choose between mismatches and indels at the boundary. This is the structural advantage of EP patterns over clip-based heuristics for STRs.

In [25]:
for ix, r, hit in zip(illustrative_idx, illustrative_reads, res_nwflex):
    print(f"Read idx={ix}   lflank={r.lflank_extent:>3}   rflank={r.rflank_extent:<3}")
    print(f"  NW-flex  {hit['cigar']}  (pos={hit['start_pos']}, score={int(hit['score'])})")
    for line in render_zoom(
        nwflex_locus.X, r.sequence, hit['start_pos'], hit['cigar'], nwflex_locus.s, nwflex_locus.e,
    ).splitlines():
        print(f"    {line}")
    print()

Read idx=0   lflank=131   rflank=1  
  NW-flex  131M27N19M  (pos=70, score=150)
    ref :  AATCTGCTGCTTATA|AACAACAACAACAACAACAACAACAACAACAACAACAACAACAAC|C
    read:  AATCTGCTGCTTATA|---------------------------AACAACAACAACAACAAC|C
                           NNNNNNNNNNNNNNNNNNNNNNNNNNN                    

Read idx=65   lflank= 66   rflank=66 
  NW-flex  66M27N84M  (pos=135, score=150)
    ref :  AATCTGCTGCTTATA|AACAACAACAACAACAACAACAACAACAACAACAACAACAACAAC|CATTGACAGGCACAA
    read:  AATCTGCTGCTTATA|---------------------------AACAACAACAACAACAAC|CATTGACAGGCACAA
                           NNNNNNNNNNNNNNNNNNNNNNNNNNN                                  

Read idx=130   lflank=  1   rflank=131
  NW-flex  1M27N149M  (pos=200, score=150)
    ref :  A|AACAACAACAACAACAACAACAACAACAACAACAACAACAACAAC|CATTGACAGGCACAA
    read:  A|---------------------------AACAACAACAACAACAAC|CATTGACAGGCACAA
             NNNNNNNNNNNNNNNNNNNNNNNNNNN                                  



### Correctness rule

We've seen two ways the alignment can go wrong on a read.

Sometimes the read gets **soft-clipped** at the boundary — a flank that was supposed to be matched ends up bracketed off as `S` instead. To catch this, the correctness rule requires the alignment to **span the repeat**: at least one reference base consumed on each side of the repeat interval (`M`/`=`/`X`/`D`/`N` ops; soft-clips and insertions don't count).

And the **decoded length of the repeat interval** ($z_{\text{decoded}}$) — the thing we're measuring — can disagree with the truth. With $N + \Delta$ motif copies of size $|R|$, the truth is $z_{\text{truth}} = (N + \Delta) \cdot |R|$ bp; we want $z_{\text{decoded}} = z_{\text{truth}}$.

A read is **correct** under a configuration when both conditions hold: it spans the repeat, and $z_{\text{decoded}} = z_{\text{truth}}$. The decoder is shared across all three configurations; its only dial is the boundary convention — an insertion exactly at the repeat boundary counts inside the repeat under BWA's convention, outside under NW-flex's.

NW-flex and BWA-MEM's no-clip version share the same boundary convention, so they decode the same CIGARs to the same repeat lengths. BWA-MEM's std version produces soft-clips that result in indeterminate repeat lengths.

In [26]:
from nwflex.simulation import is_arm_correct

bwa_zone = (locus.s, locus.e)
nw_zone  = (nwflex_locus.s, nwflex_locus.e)
truth_z_bp = hap.body_len  # (locus.N + DELTA) * |R|

print(f"Truth: z_bp = (N + Δ) · |R| = {truth_z_bp}")
print("All three configurations on the three illustrative reads, side-by-side:")
pd.DataFrame({
    ("read",    ""):        illustrative_idx,
    ("lflank",  ""):        [r.lflank_extent for r in illustrative_reads],
    ("rflank",  ""):        [r.rflank_extent for r in illustrative_reads],
    ("cigar",   "std"):     [x.cigar for x in res_std],
    ("cigar",   "no-clip"): [x.cigar for x in res_noclip],
    ("cigar",   "NW-flex"): [a["cigar"] for a in res_nwflex],
    ("score",   "std"):     [int(x.score) for x in res_std],
    ("score",   "no-clip"): [int(x.score) for x in res_noclip],
    ("score",   "NW-flex"): [int(a["score"]) for a in res_nwflex],
    ("correct", "std"):     [is_arm_correct(x.cigar, x.pos, *bwa_zone, truth_z_bp, convention="bwa") for x in res_std],
    ("correct", "no-clip"): [is_arm_correct(x.cigar, x.pos, *bwa_zone, truth_z_bp, convention="bwa") for x in res_noclip],
    ("correct", "NW-flex"): [is_arm_correct(a["cigar"], a["start_pos"], *nw_zone, truth_z_bp, convention="nwflex") for a in res_nwflex],
})

Truth: z_bp = (N + Δ) · |R| = 18
All three configurations on the three illustrative reads, side-by-side:


read lflank rflank     cigar                        score                  \
                           std    no-clip     NW-flex   std no-clip NW-flex   
0    0    131      1    146M4S  131M3I16M  131M27N19M   146     146     150   
1   65     66     66  66M3I81M   66M3I81M   66M27N84M   138     138     150   
2  130      1    131    4S146M   1M3I146M   1M27N149M   146     146     150   

  correct                  
      std no-clip NW-flex  
0   False    True    True  
1    True    True    True  
2   False    True    True

### Inequivalence of orientations

We align each read in both orientations — forward, and with both read *and* reference reverse-complemented — and credit the read if either run finds the truth. Smith-Waterman returns *a* best alignment, not all of them; when several alignments tie, the tie-breaking is deterministic and depends on the order in which DP cells are evaluated. Reversing the orientation of read and reference changes that order, so the two arms can return different (equally optimal) alignments. Running both orientations removes that order-of-evaluation artifact.

In [27]:
from nwflex.simulation import is_arm_correct, rc_to_forward_alignment, align_bwa_both_strands

# Same locus, Δ=-1 contraction. Pick read idx=4 (lflank=133, rflank=5):
# heavy left flank, tiny right flank, so fwd vs rc see very different
# alignment landscapes.
hap_dn1   = build_haplotype(locus, delta=-1)
reads_dn1 = tile_reads(hap_dn1, read_len=READ_LEN, k_min_flank=K_MIN_FLANK)
ix = 4
r_dn1 = reads_dn1[ix]
truth_dn1 = hap_dn1.body_len  # (N - 1) * |R|
ref_length = len(locus.X)

both_std = align_bwa_both_strands(locus.X, [r_dn1], no_clip=False)[0]
both_nc  = align_bwa_both_strands(locus.X, [r_dn1], no_clip=True)[0]


def verdict(pos, cigar):
    return "CORRECT" if is_arm_correct(
        cigar, pos, locus.s, locus.e, truth_dn1, convention="bwa"
    ) else "wrong  "


print(f"Haplotype: N={locus.N - 1} (locus N={locus.N}, Δ=-1)   truth z_bp={truth_dn1}")
print(f"Read idx={ix}   lflank={r_dn1.lflank_extent}   rflank={r_dn1.rflank_extent}")
print()
for mode_label, both in (("STD   ", both_std), ("NOCLIP", both_nc)):
    # Flip the rc-strand result back to forward coordinates so both
    # strands are rendered against the same reference and read.
    rc_pos, rc_cigar = rc_to_forward_alignment(both.rc.pos, both.rc.cigar, ref_length)
    for strand_label, hit_pos, hit_cigar, hit_score in (
        ("fwd", both.fwd.pos, both.fwd.cigar, both.fwd.score),
        ("rc ", rc_pos,       rc_cigar,       both.rc.score),
    ):
        print(f"  {mode_label}  {strand_label}  {verdict(hit_pos, hit_cigar)}  "
              f"{hit_cigar}  (pos={hit_pos}, AS={hit_score})")
        for line in render_zoom(
            locus.X, r_dn1.sequence, hit_pos, hit_cigar, locus.s, locus.e,
        ).splitlines():
            print(f"    {line}")
    print()

Haplotype: N=4 (locus N=5, Δ=-1)   truth z_bp=12
Read idx=4   lflank=133   rflank=5

  STD     fwd  CORRECT  133M3D17M  (pos=68, AS=145)
    ref :  AATCTGCTGCTTATA|AACAACAACAACAAC|CATTG
    read:  AATCTGCTGCTTATA|---AACAACAACAAC|CATTG
                           DDD                  
  STD     rc   wrong    145M5S  (pos=68, AS=145)
    ref :  AATCTGCTGCTTATA|AACAACAACAAC  
    read:  AATCTGCTGCTTATA|AACAACAACAAC[CATTG]

  NOCLIP  fwd  CORRECT  133M3D17M  (pos=68, AS=145)
    ref :  AATCTGCTGCTTATA|AACAACAACAACAAC|CATTG
    read:  AATCTGCTGCTTATA|---AACAACAACAAC|CATTG
                           DDD                  
  NOCLIP  rc   wrong    147M3I  (pos=68, AS=145)
    ref :  AATCTGCTGCTTATA|AACAACAACAACAA---
    read:  AATCTGCTGCTTATA|AACAACAACAACCATTG
                                         III



## First comparison — length variation only

We run the simulation above with the haplotype flanks left untouched and the repeat count varying over $N + \Delta$ for a small range of $\Delta$. We tabulate correctness for each method as a function of flank extent and $\Delta$ and present the result as a heatmap with one panel per method.

**Expected result.** NW-flex is uniformly correct as long as the read has any flank extent. BWA-MEM at standard parameters fails along the boundary even though the haplotype differs from the reference only in the repeat count. The no-clip arm recovers some of these cases but not all.

In [28]:
# TODO: sweep (flank_extent, Δ) → correctness for each arm

In [29]:
# TODO: 3-panel heatmap (one per arm), x=Δ, y=flank_extent

## Second comparison — a single SNV in the flank

We repeat the first comparison with one change: the haplotype carries a single SNV at a fixed position. We choose for our example a single base change in the left flank, one base outside the repeat boundary. The locus, the read tiling, and the alignment configurations are unchanged.

**Expected result.** The same heatmap now reads differently. NW-flex remains correct in the regime where it has flank overhang, but not always — sometimes the local sequence and variant have a higher-value alignment. The no-clip arm — the one that recovered the easy cases above — now fails on the reads that cross the SNV.

In [30]:
# TODO: same sweep with one boundary-adjacent SNV injected into the haplotype

In [31]:
# TODO: 3-panel heatmap, same axes as comparison 1

## Third comparison — compound repeat

The third comparison changes the locus structure. The reference is built from two adjacent repeat motifs joined by a short interrupting sequence,

$$X = A \cdot R_1^{N_1} \cdot M \cdot R_2^{N_2} \cdot B,$$

and the haplotype varies both counts independently. The reads, the alignment methods, and the correctness rule carry over.

**Expected result.** The EP pattern for two repeat blocks enumerates the product of allowed counts in a single pass, so NW-flex is correct everywhere on the $(\Delta_1, \Delta_2)$ grid by construction. BWA-MEM is correct on part of the grid; the size and shape of the failure region depends on motif similarity, the length of the interrupting sequence, and the absolute counts.

In [32]:
# TODO: build compound locus + haplotype grid over (Δ₁, Δ₂); align all three arms

In [33]:
# TODO: (Δ₁, Δ₂) heatmap per arm

## Summary

Across all three comparisons, NW-flex matches the haplotype's repeat-region length whenever the read has flank overhang on both sides. BWA-MEM at standard parameters loses the flank-adjacent reads to soft-clipping; raising the clip penalty recovers the no-variant cases but not the cases with a boundary-adjacent SNV. On compound loci, NW-flex is correct by construction across the full $(\Delta_1, \Delta_2)$ grid while BWA-MEM degrades along a method-dependent failure surface.

The headline is structural: when the only difference between haplotype and reference is repeat count, NW-flex's EP pattern is guaranteed to find the optimum, while a clip-based aligner is at the mercy of its scoring tradeoffs at the boundary.